In [1]:
# Resolve project imports and data paths when this notebook runs in analysis/.
from pathlib import Path
import os
import sys

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "run.py").is_file()
     and (folder / "GNN" / "GNN_utils.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start this notebook from within the ligo-opt repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
from tqdm import tqdm

from utils.finesse_base import base_kat
from oracles import finesse_sim
from utils.calc import _get_power_for_pd_name
from GNN.GNN_utils import kat_manipulation
from PSOoptimizer import main_wrapper_PSO
from algorithm import SteppablePSO

/home/xuesi.ma/.conda/envs/ligoopt/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Nominal design point and PSO search bounds (+/- 500 m around each ROC)
ITM_ROC = -1934
ETM_ROC = 2245
port = 'p_ITM_p2_i'

start_ITM_Roc, end_ITM_Roc = ITM_ROC - 1500, ITM_ROC + 500
start_ETM_Roc, end_ETM_Roc = ETM_ROC - 500, ETM_ROC + 1500

## The two all-Finesse cost functions

In [3]:
# COST A -- gradient-style Finesse cost (the one from main_wrapper_PSO):
# every single evaluation runs 5 REAL Finesse simulations (the nominal 'D' run
# plus +/- Delta ITM and +/- Delta ETM ROC perturbations) and returns the
# finite-difference gain-variation cost from calc_cost. So a full PSO on this
# function costs 5 * pop * iterations Finesse sims (~3 s each).
cost_fn_finesse_grad = main_wrapper_PSO(base_kat, port)

print('cost at nominal point:', cost_fn_finesse_grad((ITM_ROC, ETM_ROC)))

cost at nominal point: 0.01775646096190691


In [4]:
# COST B -- fast direct Finesse cost: exactly ONE Finesse simulation per
# evaluation, cost = 1 / circulating power at `port`. Same model construction
# as the nominal 'D' run of COST A (aperture maps applied, eigenmode q solved,
# locks keep ETM.phi tuned).
def cost_fn_finesse_fast(params):
    ITM_Roc, ETM_Roc = params

    run_kat = kat_manipulation(ITM_Roc, ETM_Roc, base_kat)
    pd_names, pd_powers, _, _ = finesse_sim(run_kat)

    power = _get_power_for_pd_name(port, pd_names, pd_powers)
    if power <= 0:
        # Unstable cavity or lost lock -> worst possible cost.
        return np.inf
    return 1.0 / power

c = cost_fn_finesse_fast((ITM_ROC, ETM_ROC))
print('cost at nominal point:', c, '-> circulating power', 1.0 / c, 'W')

cost at nominal point: 0.003545215320794783 -> circulating power 282.07031435704596 W


In [5]:
# FULL PSO #1 -- every particle scored by the 5-sim gradient-style Finesse cost.
# Budget check: POP_GRAD * (ITER_GRAD + 1) * 5 sims x ~3 s each.
POP_GRAD, ITER_GRAD = 10, 40

pso_grad = SteppablePSO(
    func=cost_fn_finesse_grad, n_dim=2,
    pop=POP_GRAD, max_iter=ITER_GRAD,
    lb=[start_ITM_Roc, start_ETM_Roc], ub=[end_ITM_Roc, end_ETM_Roc],  # fixed: was [end_ETM_Roc, end_ETM_Roc]
    w=0.9, c1=2.0, c2=0.4, verbose=True,
)

for i in tqdm(range(ITER_GRAD), desc='PSO (Finesse, 5-sim grad cost)'):
    # if i == ITER_GRAD // 2:
    #     pso_grad.set_func(cost_fn_finesse_fast)  # e.g. switch to the faster cost partway through
    pso_grad.step()

PSO (Finesse, 5-sim grad cost):   5%|▌         | 2/40 [07:59<2:31:48, 239.69s/it]
/home/xuesi.ma/.conda/envs/ligoopt/lib/python3.13/site-packages/sko/PSO.py:33: SyntaxWarning: invalid escape sequence '\h'
  + c_{g}r_{2j}(t)[\hat{y}_{j}(t) − x_{ij}(t)]


KeyboardInterrupt: 

In [ ]:
print('Best parameters found (ITM_ROC, ETM_ROC): ', pso_grad.gbest_x)
print('Best cost found: ', pso_grad.gbest_y)
print('gbest history: ', pso_grad.gbest_y_hist)

In [ ]:
# FULL PSO #2 -- identical setup, but every particle is scored by ONE Finesse
# simulation (inverse circulating power). ~8x cheaper per evaluation than the
# 5-sim gradient cost, so a much bigger swarm fits in the same wall-clock budget.
POP_FAST, ITER_FAST = 30, 100

pso_fast = SteppablePSO(
    func=cost_fn_finesse_fast, n_dim=2,
    pop=POP_FAST, max_iter=ITER_FAST,
    lb=[start_ITM_Roc, start_ETM_Roc], ub=[end_ITM_Roc, end_ETM_Roc],
    w=0.9, c1=2.0, c2=0.4, verbose=True,
)

for i in tqdm(range(ITER_FAST), desc='PSO (Finesse, 1-sim power cost)'):
    pso_fast.step()

In [ ]:
best_power = 1.0 / pso_fast.gbest_y if np.isfinite(pso_fast.gbest_y) else 0.0
print('Best parameters found (ITM_ROC, ETM_ROC): ', pso_fast.gbest_x)
print('Best cost found: ', pso_fast.gbest_y, '-> circulating power', best_power, 'W')
print('gbest history: ', pso_fast.gbest_y_hist)